In [2]:
import pandas as pd

s01 = pd.read_csv(
    "../data/staging/standings/s01_standings.csv"
)

s04 = pd.read_csv(
    "../data/staging/rosters/s04_rosters.csv"
)

s08 = pd.read_csv(
    "../data/staging/team_payroll/s08_team_payroll.csv"
)

s15 = pd.read_csv( "../data/staging/front_office/S15_GM_Tenures.csv")

s16 = pd.read_csv( "../data/staging/ownership/S16_Ownership_Tenures.csv")

print("S01:", s01.shape, s01.columns.tolist())
print("S04:", s04.shape, s04.columns.tolist())
print("S08:", s08.shape, s08.columns.tolist())
print("S15:", s15.shape, s15.columns.tolist())
print("S16:", s16.shape, s16.columns.tolist())

S01: (776, 11) ['team', 'W', 'L', 'W/L%', 'GB', 'PS/G', 'PA/G', 'SRS', 'conference', 'season', 'year']
S04: (12810, 25) ['PLAYER_ID', 'PLAYER_NAME', 'TEAM_ID', 'TEAM_ABBREVIATION', 'AGE', 'PLAYER_HEIGHT', 'PLAYER_HEIGHT_INCHES', 'PLAYER_WEIGHT', 'COLLEGE', 'COUNTRY', 'DRAFT_YEAR', 'DRAFT_ROUND', 'DRAFT_NUMBER', 'GP', 'PTS', 'REB', 'AST', 'NET_RATING', 'OREB_PCT', 'DREB_PCT', 'USG_PCT', 'TS_PCT', 'AST_PCT', 'season', 'year']
S08: (776, 6) ['Team_Name', 'Team_Payroll', 'Salary_Cap_Hit', 'Salary_Cap_Space', 'Season', 'Year']
S15: (181, 5) ['executive_name', 'start_date', 'end_date', 'franchise_id', 'franchise_name']
S16: (70, 7) ['owner_name', 'team_id', 'start_season', 'end_season', 'start_year', 'end_year', 'source']


In [3]:
s01["team_clean"] = s01["team"].str.replace(
    r"\*$", "", regex=True
).str.strip()

In [4]:
# See whether a numeric NBA team ID maps to
# multiple abbreviations over time.
display(
    s04.groupby("TEAM_ID")["TEAM_ABBREVIATION"]
       .unique()
)

# See all franchise IDs used in GM tenures.
print(sorted(s15["franchise_id"].dropna().unique()))

# See all team names used in payroll.
print(sorted(s08["Team_Name"].dropna().unique()))

TEAM_ID
1610612737              [ATL]
1610612738              [BOS]
1610612739              [CLE]
1610612740    [NOH, NOK, NOP]
1610612741              [CHI]
1610612742              [DAL]
1610612743              [DEN]
1610612744              [GSW]
1610612745              [HOU]
1610612746              [LAC]
1610612747              [LAL]
1610612748              [MIA]
1610612749              [MIL]
1610612750              [MIN]
1610612751         [NJN, BKN]
1610612752              [NYK]
1610612753              [ORL]
1610612754              [IND]
1610612755              [PHI]
1610612756              [PHX]
1610612757              [POR]
1610612758              [SAC]
1610612759              [SAS]
1610612760         [SEA, OKC]
1610612761              [TOR]
1610612762              [UTA]
1610612763         [VAN, MEM]
1610612764              [WAS]
1610612765              [DET]
1610612766         [CHH, CHA]
Name: TEAM_ABBREVIATION, dtype: object

['ATL', 'BOS', 'BRK', 'CHI', 'CHO', 'CLE', 'DAL', 'DEN', 'DET', 'GSW', 'HOU', 'IND', 'LAC', 'LAL', 'MEM', 'MIA', 'MIL', 'MIN', 'NOP', 'NYK', 'OKC', 'ORL', 'PHI', 'PHO', 'POR', 'SAC', 'SAS', 'TOR', 'UTA', 'WAS']
['76ers', 'Bobcats', 'Bucks', 'Bulls', 'Cavaliers', 'Celtics', 'Clippers', 'Grizzlies', 'Hawks', 'Heat', 'Hornets', 'Jazz', 'Kings', 'Knicks', 'Lakers', 'Magic', 'Mavericks', 'Nets', 'Nuggets', 'Pacers', 'Pelicans', 'Pistons', 'Raptors', 'Rockets', 'Spurs', 'Suns', 'SuperSonics', 'Thunder', 'Timberwolves', 'Trail Blazers', 'Warriors', 'Wizards']


In [5]:
# Create one record per numeric team ID, season,
# and historical abbreviation.
team_aliases = (
    s04[
        ["TEAM_ID", "TEAM_ABBREVIATION", "season"]
    ]
    .drop_duplicates()
    .sort_values(["season", "TEAM_ID"])
    .reset_index(drop=True)
)

print("Distinct team-season-abbreviation records:", len(team_aliases))
print("Distinct numeric team IDs:", team_aliases["TEAM_ID"].nunique())

# Check whether any team-season has multiple abbreviations.
conflicts = (
    team_aliases
    .groupby(["TEAM_ID", "season"])
    .size()
)

print("Team-season abbreviation conflicts:", (conflicts > 1).sum())

Distinct team-season-abbreviation records: 776
Distinct numeric team IDs: 30
Team-season abbreviation conflicts: 0


In [6]:
# Distinct full team names in standings
standings_names = sorted(
    s01["team"]
    .str.replace(r"\*$", "", regex=True)
    .str.strip()
    .unique()
)

# Distinct short team names in payroll
payroll_names = sorted(
    s08["Team_Name"].dropna().unique()
)

print("S01 full team names:")
print(standings_names)

print("\nS08 short team names:")
print(payroll_names)

S01 full team names:
['Atlanta Hawks', 'Boston Celtics', 'Brooklyn Nets', 'Charlotte Bobcats', 'Charlotte Hornets', 'Chicago Bulls', 'Cleveland Cavaliers', 'Dallas Mavericks', 'Denver Nuggets', 'Detroit Pistons', 'Golden State Warriors', 'Houston Rockets', 'Indiana Pacers', 'Los Angeles Clippers', 'Los Angeles Lakers', 'Memphis Grizzlies', 'Miami Heat', 'Milwaukee Bucks', 'Minnesota Timberwolves', 'New Jersey Nets', 'New Orleans Hornets', 'New Orleans Pelicans', 'New Orleans/Oklahoma City Hornets', 'New York Knicks', 'Oklahoma City Thunder', 'Orlando Magic', 'Philadelphia 76ers', 'Phoenix Suns', 'Portland Trail Blazers', 'Sacramento Kings', 'San Antonio Spurs', 'Seattle SuperSonics', 'Toronto Raptors', 'Utah Jazz', 'Vancouver Grizzlies', 'Washington Wizards']

S08 short team names:
['76ers', 'Bobcats', 'Bucks', 'Bulls', 'Cavaliers', 'Celtics', 'Clippers', 'Grizzlies', 'Hawks', 'Heat', 'Hornets', 'Jazz', 'Kings', 'Knicks', 'Lakers', 'Magic', 'Mavericks', 'Nets', 'Nuggets', 'Pacers', 'Pe

In [7]:
full_to_short = {
    "Atlanta Hawks": "Hawks",
    "Boston Celtics": "Celtics",
    "Brooklyn Nets": "Nets",
    "Charlotte Bobcats": "Bobcats",
    "Charlotte Hornets": "Hornets",
    "Chicago Bulls": "Bulls",
    "Cleveland Cavaliers": "Cavaliers",
    "Dallas Mavericks": "Mavericks",
    "Denver Nuggets": "Nuggets",
    "Detroit Pistons": "Pistons",
    "Golden State Warriors": "Warriors",
    "Houston Rockets": "Rockets",
    "Indiana Pacers": "Pacers",
    "Los Angeles Clippers": "Clippers",
    "Los Angeles Lakers": "Lakers",
    "Memphis Grizzlies": "Grizzlies",
    "Miami Heat": "Heat",
    "Milwaukee Bucks": "Bucks",
    "Minnesota Timberwolves": "Timberwolves",
    "New Jersey Nets": "Nets",
    "New Orleans Hornets": "Hornets",
    "New Orleans Pelicans": "Pelicans",
    "New Orleans/Oklahoma City Hornets": "Hornets",
    "New York Knicks": "Knicks",
    "Oklahoma City Thunder": "Thunder",
    "Orlando Magic": "Magic",
    "Philadelphia 76ers": "76ers",
    "Phoenix Suns": "Suns",
    "Portland Trail Blazers": "Trail Blazers",
    "Sacramento Kings": "Kings",
    "San Antonio Spurs": "Spurs",
    "Seattle SuperSonics": "SuperSonics",
    "Toronto Raptors": "Raptors",
    "Utah Jazz": "Jazz",
    "Vancouver Grizzlies": "Grizzlies",
    "Washington Wizards": "Wizards",
}

In [8]:
# Prepare standings
standings = s01[["season", "team"]].copy()

standings["full_team_name"] = (
    standings["team"]
    .str.replace(r"\*$", "", regex=True)
    .str.strip()
)

standings["short_team_name"] = (
    standings["full_team_name"].map(full_to_short)
)

# Prepare payroll
payroll = s08[["Season", "Team_Name"]].copy()

payroll = payroll.rename(
    columns={
        "Season": "season",
        "Team_Name": "payroll_team_name"
    }
)

payroll["payroll_team_name"] = (
    payroll["payroll_team_name"].str.strip()
)

In [9]:
name_mapping = standings.merge(
    payroll,
    left_on=["season", "short_team_name"],
    right_on=["season", "payroll_team_name"],
    how="left",
    validate="one_to_one",
    indicator=True
)

print(name_mapping["_merge"].value_counts())

display(
    name_mapping.loc[
        name_mapping["_merge"] != "both"
    ]
)

_merge
both          776
left_only       0
right_only      0
Name: count, dtype: int64


,season,team,full_team_name,short_team_name,payroll_team_name,_merge


In [10]:
abbrev_to_full = {
    "ATL": "Atlanta Hawks",
    "BOS": "Boston Celtics",
    "BKN": "Brooklyn Nets",
    "CHA": "Charlotte Hornets",
    "CHH": "Charlotte Hornets",
    "CHI": "Chicago Bulls",
    "CLE": "Cleveland Cavaliers",
    "DAL": "Dallas Mavericks",
    "DEN": "Denver Nuggets",
    "DET": "Detroit Pistons",
    "GSW": "Golden State Warriors",
    "HOU": "Houston Rockets",
    "IND": "Indiana Pacers",
    "LAC": "Los Angeles Clippers",
    "LAL": "Los Angeles Lakers",
    "MEM": "Memphis Grizzlies",
    "MIA": "Miami Heat",
    "MIL": "Milwaukee Bucks",
    "MIN": "Minnesota Timberwolves",
    "NJN": "New Jersey Nets",
    "NOH": "New Orleans Hornets",
    "NOK": "New Orleans/Oklahoma City Hornets",
    "NOP": "New Orleans Pelicans",
    "NYK": "New York Knicks",
    "OKC": "Oklahoma City Thunder",
    "ORL": "Orlando Magic",
    "PHI": "Philadelphia 76ers",
    "PHX": "Phoenix Suns",
    "POR": "Portland Trail Blazers",
    "SAC": "Sacramento Kings",
    "SAS": "San Antonio Spurs",
    "SEA": "Seattle SuperSonics",
    "TOR": "Toronto Raptors",
    "UTA": "Utah Jazz",
    "VAN": "Vancouver Grizzlies",
    "WAS": "Washington Wizards",
}

In [11]:
# Work from the validated S04 franchise-season backbone.
aliases = team_aliases.copy()

aliases["full_team_name"] = (
    aliases["TEAM_ABBREVIATION"].map(abbrev_to_full)
)

# Charlotte used the Bobcats name from 2004-05 through 2013-14.
charlotte_bobcats = (
    (aliases["TEAM_ABBREVIATION"] == "CHA") &
    (aliases["season"].str[:4].astype(int).between(2004, 2013))
)

aliases.loc[
    charlotte_bobcats, "full_team_name"
] = "Charlotte Bobcats"

In [12]:
c02_aliases = aliases.merge(
    name_mapping[
        ["season", "full_team_name", "short_team_name"]
    ],
    on=["season", "full_team_name"],
    how="left",
    validate="one_to_one",
    indicator=True
)

print(c02_aliases["_merge"].value_counts())

display(
    c02_aliases.loc[
        c02_aliases["_merge"] != "both"
    ]
)

_merge
both          776
left_only       0
right_only      0
Name: count, dtype: int64


,TEAM_ID,TEAM_ABBREVIATION,season,full_team_name,short_team_name,_merge


In [13]:
c02_aliases = (
    c02_aliases
    .rename(columns={
        "TEAM_ID": "franchise_id",
        "TEAM_ABBREVIATION": "team_abbreviation"
    })
    [[
        "franchise_id",
        "season",
        "team_abbreviation",
        "full_team_name",
        "short_team_name"
    ]]
    .sort_values(["season", "franchise_id"])
    .reset_index(drop=True)
)

In [14]:
assert len(c02_aliases) == 776

assert not c02_aliases.duplicated(
    ["franchise_id", "season"]
).any()

assert c02_aliases.notna().all().all()

print("C02 alias table validated.")

C02 alias table validated.


In [15]:
c02_franchises = (
    c02_aliases
    .sort_values("season")
    .drop_duplicates("franchise_id", keep="last")
    [[
        "franchise_id",
        "full_team_name",
        "team_abbreviation"
    ]]
    .rename(columns={
        "full_team_name": "current_team_name",
        "team_abbreviation": "current_abbreviation"
    })
    .sort_values("franchise_id")
    .reset_index(drop=True)
)

assert len(c02_franchises) == 30
assert c02_franchises["franchise_id"].is_unique

display(c02_franchises.head())

,franchise_id,current_team_name,current_abbreviation
0,1610612737,Atlanta Hawks,ATL
1,1610612738,Boston Celtics,BOS
2,1610612739,Cleveland Cavaliers,CLE
3,1610612740,New Orleans Pelicans,NOP
4,1610612741,Chicago Bulls,CHI


In [17]:
c02_aliases.to_csv(
    "../data/canonical/franchise_season_aliases.csv",
    index=False
)

c02_franchises.to_csv(
    "../data/canonical/franchises.csv",
    index=False
)

print("C02 canonical files exported.")

C02 canonical files exported.
